In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
from scipy.spatial import distance

In [2]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
FaceLandmarkerResult = mp.tasks.vision.FaceLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode
model_path = 'face_landmarker.task'

def result_callback(result, output_image: mp.Image, timestamp_ms: int):
    print('face landmarker result: {}'.format(result))

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    output_facial_transformation_matrixes=True,
    #result_callback=result_callback
    )

landmarker = FaceLandmarker.create_from_options(options)

In [3]:
HEAD_POSE_LANDMARKS = [1, 33, 263, 61, 291, 199]
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
blink_history = []

def draw_landmarks(frame, points):
    for p in points:
        cv2.circle(frame, tuple(p), 1, (0,255,0), -1)

def draw_eye_mesh(frame, points):
    for idx in LEFT_EYE:
        cv2.circle(frame, tuple(points[idx]), 2, (255,0,0), -1)

    for idx in RIGHT_EYE:
        cv2.circle(frame, tuple(points[idx]), 2, (255,0,0), -1)

In [4]:
def get_head_pose(matrix):
    R = matrix[:3,:3]

    sy = math.sqrt(R[0,0]*R[0,0] + R[1,0]*R[1,0])

    singular = sy < 1e-6

    if not singular:

        pitch = math.atan2(R[2,1], R[2,2])
        yaw = math.atan2(-R[2,0], sy)
        roll = math.atan2(R[1,0], R[0,0])

    else:

        pitch = math.atan2(-R[1,2], R[1,1])
        yaw = math.atan2(-R[2,0], sy)
        roll = 0

    pitch = math.degrees(pitch)
    yaw = math.degrees(yaw)
    roll = math.degrees(roll)

    return yaw, pitch, roll

def compute_EAR(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    ear = (A + B) / (2.0 * C)

    return ear

def compute_blink_rate(timestamp, blink_counter):

    blink_history.append((timestamp, blink_counter))

    while blink_history and timestamp - blink_history[0][0] > 60:
        blink_history.pop(0)

    if len(blink_history) > 1:
        rate = blink_history[-1][1] - blink_history[0][1]
    else:
        rate = 0

    return rate

class EMAFilter:
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.value = None

    def update(self, x):
        if self.value is None:
            self.value = x
        else:
            self.value = self.alpha*x + (1-self.alpha)*self.value
        return self.value

In [5]:
def decision_model(yaw, pitch, ear, closed_frames):

    if closed_frames > 20:
        return "Drowsy"

    if abs(yaw) > 25 or abs(pitch) > 20:
        return "Distracted"

    if ear > 0.23:
        return "Focused"

    return "Neutral"

In [6]:
ear_filter = EMAFilter()
yaw_filter = EMAFilter()
pitch_filter = EMAFilter()
EAR_THRESH = 0.21
blink_counter = 0
closed_frames = 0

SHOW_LANDMARKS = True
SHOW_EYE_MESH = True

try:
    cap = cv2.VideoCapture(0)
    timestamp = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.flip(frame,1)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame
        )

        result = landmarker.detect_for_video(mp_image, timestamp)

        timestamp += 1

        if result.face_landmarks:

            landmarks = result.face_landmarks[0]

            h,w,_ = frame.shape

            points = []

            for lm in landmarks:
                x = int(lm.x*w)
                y = int(lm.y*h)
                points.append((x,y))

            points = np.array(points)

            if SHOW_LANDMARKS:
                draw_landmarks(frame, points)

            if SHOW_EYE_MESH:
                draw_eye_mesh(frame, points)

            left_eye = points[LEFT_EYE]
            right_eye = points[RIGHT_EYE]

            leftEAR = compute_EAR(left_eye)
            rightEAR = compute_EAR(right_eye)

            ear = (leftEAR + rightEAR)/2
            ear = ear_filter.update(ear)

            if ear < EAR_THRESH:
                closed_frames += 1
            else:
                if closed_frames > 2:
                    blink_counter += 1
                closed_frames = 0

            if result.facial_transformation_matrixes:
                matrix = np.array(result.facial_transformation_matrixes[0]).reshape(4,4)
                yaw, pitch, roll = get_head_pose(matrix)
            else:
                yaw, pitch, roll = None, None, None

            yaw = yaw_filter.update(yaw)
            pitch = pitch_filter.update(pitch)

            attention_state = decision_model(yaw, pitch, ear, closed_frames)

            cv2.putText(frame,f"EAR: {ear:.2f}",(20,40),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)

            cv2.putText(frame,f"Yaw: {yaw:.1f}",(20,70),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,0,0),2)

            cv2.putText(frame,f"Pitch: {pitch:.1f}",(20,100),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,0,0),2)

            cv2.putText(frame,f"Blinks: {blink_counter}",(20,130),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,255),2)

            cv2.putText(frame,f"Attention: {attention_state}",(20,160),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,0),2)

        cv2.imshow("Face Monitoring", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # Press 'Esc' to exit
            break

finally:
    cap.release()
    cv2.destroyAllWindows()